# RSNA Knee Abnormality Detection — Phase 4F.4 documented-AMP unlabeled image SSL canary

Internet-off Kaggle notebook. **Do not enable internet. GPU (Tesla T4) is required.**
This notebook runs the **authorized 64-study / 40-applied-step documented-AMP** image-only SimSiam canary.
It does **not** launch the full 4,349-study SSL job.
`UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED=false`.
`UNLABELED_IMAGE_SSL_CANARY_V4_EXECUTION_ALLOWED=true`.
Do **not** fine-tune on gold, do **not** run OOF, do **not** infer test, and
do **not** write `/kaggle/working/submission.csv`.

Create a **new** kernel slug `mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v4`.
Do **not** overwrite `mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v1` versions 1 or 2.
Do **not** overwrite frozen kernels (gold-pilot v7, constrained-canary v3,
gold-constrained-v14 v1, gold58-image-oof-v1 v1, gold58-image-oof-v2 v1,
gold58-submission-rehearsal-v1 v1, unlabeled-image-ssl-v1, unlabeled-image-ssl-canary-v1 versions 1–2, unlabeled-image-ssl-canary-v2 version 1, unlabeled-image-ssl-canary-v3 version 1, notebooks 07/09/10/11/12/13/14/15/16).

`HARD_WEAK_LABEL_TRAINING_ALLOWED=false`.
`SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=false`.
`SECOND_READER_SCORE_BEARING_ALLOWED=false`.
`GOLD58_SSL_OOF_EXECUTION_ALLOWED=false`.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_unlabeled_image_ssl_canary_v4"
MARKERS = ("knee/config.py", "knee/phase4f_ssl_canary_v4.py")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TORCH_HOME"] = str(KAGGLE_WORKING / "torch_cache")


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks", "vendor", "third_party"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md", "literature/ontology.yaml"):
        src = src_root / filename
        if src.is_file():
            dest_file = dest / filename
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest_file)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips = []
    dirs = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG)
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source():
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips = []
    dirs = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        _safe_extract(zips[0], SRC_DST)
        nested = [p for p in SRC_DST.rglob("knee") if (p.parent / "knee" / "phase4f_ssl_canary_v4.py").is_file()]
        if _looks_like_source(SRC_DST):
            return SRC_DST
        if nested:
            return nested[0].parent
    if dirs:
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
print("source_root=" + str(SOURCE_ROOT))
if SOURCE_ROOT is None:
    raise SystemExit("SOURCE_NOT_FOUND: attach mahdimzafari/cursor-knee-source")
sys.path.insert(0, str(SOURCE_ROOT))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("output_dir=" + str(OUTPUT_DIR))


In [ ]:
import json
from pathlib import Path

from knee.kaggle_runtime import KaggleLayout, preflight
from knee.phase4d_readiness import SECOND_READER_SCORE_BEARING_ALLOWED
from knee.phase4f_ssl import (
    GOLD58_SSL_OOF_EXECUTION_ALLOWED,
    UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED,
)
from knee.phase4f_ssl_canary_v4 import (
    KERNEL_SLUG,
    UNLABELED_IMAGE_SSL_CANARY_V4_EXECUTION_ALLOWED,
    run_phase4f_ssl_canary_v4,
)
from knee.report_labels import (
    HARD_WEAK_LABEL_TRAINING_ALLOWED,
    SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED,
)

print("kernel_slug=" + KERNEL_SLUG)
print("kernel_slug_literal=mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v4")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-gold-pilot")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-constrained-canary")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-gold-constrained-v14")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-gold58-image-oof-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-gold58-image-oof-v2")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-gold58-submission-rehearsal-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-unlabeled-image-ssl-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v2")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-unlabeled-image-ssl-canary-v3")
print("notebook_07_executed=false")
print("notebook_09_executed=false")
print("notebook_10_executed=false")
print("notebook_11_executed=false")
print("notebook_12_executed=false")
print("notebook_13_executed=false")
print("notebook_15_executed=false")
print("notebook_16_executed=false")
print("UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED=" + str(UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED).lower())
print("UNLABELED_IMAGE_SSL_CANARY_V4_EXECUTION_ALLOWED=" + str(UNLABELED_IMAGE_SSL_CANARY_V4_EXECUTION_ALLOWED).lower())
print("GOLD58_SSL_OOF_EXECUTION_ALLOWED=" + str(GOLD58_SSL_OOF_EXECUTION_ALLOWED).lower())
print("HARD_WEAK_LABEL_TRAINING_ALLOWED=" + str(HARD_WEAK_LABEL_TRAINING_ALLOWED).lower())
print("SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=" + str(SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED).lower())
print("SECOND_READER_SCORE_BEARING_ALLOWED=" + str(SECOND_READER_SCORE_BEARING_ALLOWED).lower())
print("DO_NOT_SUBMIT")
print("DO_NOT_LAUNCH_FULL_SSL")

assert HARD_WEAK_LABEL_TRAINING_ALLOWED is False
assert SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED is False
assert SECOND_READER_SCORE_BEARING_ALLOWED is False
assert UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED is False
assert GOLD58_SSL_OOF_EXECUTION_ALLOWED is False
assert UNLABELED_IMAGE_SSL_CANARY_V4_EXECUTION_ALLOWED is True

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("gpu_name=" + gpu_name)
if gpu_name == "cpu":
    raise SystemExit("SSL_CANARY_GPU_PREFLIGHT_FAIL:cuda_required")
if "Tesla T4" not in gpu_name:
    raise SystemExit("SSL_CANARY_GPU_PREFLIGHT_FAIL:expected_tesla_t4")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

from knee.phase4f_ssl_canary_v4 import (
    STATUS_GPU_PREFLIGHT,
    load_canary_config,
    run_canary_gpu_preflight,
)

gpu_cfg = load_canary_config(str(Path(SOURCE_ROOT) / "configs" / "unlabeled_image_simsiam_ssl_canary_v4.yaml"))
gpu_pre = run_canary_gpu_preflight(
    cfg=gpu_cfg,
    require_tesla_t4=True,
    require_real_resnet18=True,
)
print("gpu_preflight_status=" + str(gpu_pre.get("status")))
print("gpu_preflight_ok=" + str(gpu_pre.get("ok")).lower())
print("gpu_preflight_eigvalsh=" + json.dumps(gpu_pre.get("eigvalsh") or {}, sort_keys=True))
print("gpu_preflight_loss_dtype=" + str(gpu_pre.get("loss_dtype")))
print("gpu_preflight_loss_finite=" + str(gpu_pre.get("loss_finite")).lower())
print("gpu_preflight_grad_norm_finite=" + str(gpu_pre.get("grad_norm_finite")).lower())
print("gpu_preflight_applied_optimizer_steps=" + str(gpu_pre.get("applied_optimizer_steps")))
print("gpu_preflight_skipped_attempts=" + str(gpu_pre.get("skipped_attempts")))
print("gpu_preflight_scaler_initial_scale=" + str(gpu_pre.get("scaler_initial_scale")))
print("gpu_preflight_scaler_final_scale=" + str(gpu_pre.get("scaler_final_scale")))
print("gpu_preflight_peak_gpu_memory_bytes=" + str(gpu_pre.get("peak_gpu_memory_bytes")))
print("gpu_preflight_batch_shape=" + str(gpu_pre.get("preflight_batch_shape")))
print("gpu_preflight_precision_mode=" + str(gpu_pre.get("precision_mode")))
print("gpu_preflight_amp_call_order=" + json.dumps(gpu_pre.get("amp_call_order") or []))
print("gpu_preflight_amp_ok=" + str(gpu_pre.get("amp_preflight_ok")).lower())
print("gpu_preflight_fp32_fallback_used=" + str(gpu_pre.get("fp32_fallback_used")).lower())
print("gpu_preflight_exception_diagnostics=" + json.dumps(gpu_pre.get("exception_diagnostics") or {}, sort_keys=True)[:2000])
print("gpu_preflight_n_real_unlabelled_studies_pixels_opened=" + str(gpu_pre.get("n_real_unlabelled_studies_pixels_opened")))
print("n_report_text_rows_processed=" + str(gpu_pre.get("n_report_text_rows_processed", 0)))
print("n_second_reader_rows_processed=" + str(gpu_pre.get("n_second_reader_rows_processed", 0)))
print("DO_NOT_SUBMIT")
print("DO_NOT_LAUNCH_FULL_SSL")
if not gpu_pre.get("ok"):
    (OUTPUT_DIR / "sanitized").mkdir(parents=True, exist_ok=True)
    fail_payload = {
        "status": STATUS_GPU_PREFLIGHT,
        "ok": False,
        "skip_reason": gpu_pre.get("skip_reason"),
        "gpu_preflight": {
            "status": gpu_pre.get("status"),
            "gpu_name": gpu_pre.get("gpu_name"),
            "eigvalsh": gpu_pre.get("eigvalsh"),
            "precision_mode": gpu_pre.get("precision_mode"),
            "amp_call_order": gpu_pre.get("amp_call_order"),
            "exception_diagnostics": gpu_pre.get("exception_diagnostics"),
            "amp_exception_diagnostics": gpu_pre.get("amp_exception_diagnostics"),
            "fp32_exception_diagnostics": gpu_pre.get("fp32_exception_diagnostics"),
        },
        "n_real_unlabelled_studies_pixels_opened": 0,
        "n_report_text_rows_processed": 0,
        "n_second_reader_rows_processed": 0,
        "report_text_never_read": True,
        "full_ssl_run_authorized": False,
    }
    (OUTPUT_DIR / "sanitized" / "phase4f_ssl_canary_v4.json").write_text(
        json.dumps(fail_payload, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    raise SystemExit("SSL_CANARY_GPU_PREFLIGHT_FAIL:" + str(gpu_pre.get("skip_reason")))

layout = KaggleLayout(output_dirname="rsna_unlabeled_image_ssl_canary_v4")
pre = preflight(layout=layout, output_dir=OUTPUT_DIR)
print(pre.message)
print("data_source=" + str(pre.data_source))
if not pre.ok or pre.data_root is None:
    raise SystemExit("DATA_NOT_AVAILABLE: attach rsna-knee-abnormality-detection")

cfg_path = str(Path(SOURCE_ROOT) / "configs" / "unlabeled_image_simsiam_ssl_canary_v4.yaml")
mid = run_phase4f_ssl_canary_v4(
    data_root=pre.data_root,
    output_dir=OUTPUT_DIR,
    config_path=cfg_path,
    execute_training=True,
    allow_tiny_encoder_for_tests=False,
    internet_enabled=False,
    run_oof=False,
    run_full_unlabeled=False,
    segment="train_to_mid",
)
print("mid_status=" + str(mid.get("status")))
print("mid_completed_step=" + str((mid.get("training") or {}).get("completed_step")))
print("DO_NOT_SUBMIT")
if str(mid.get("status")) != "SSL_CANARY_MID_CHECKPOINT":
    raise SystemExit("CANARY_MID_FAIL:" + str(mid.get("status")) + ":" + str(mid.get("skip_reason")))
if Path("/kaggle/working/submission.csv").exists():
    raise SystemExit("REFUSING_WORKING_ROOT_SUBMISSION_CSV")


In [ ]:
import json
from pathlib import Path

from knee.kaggle_runtime import KaggleLayout, preflight
from knee.phase4f_ssl import UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED
from knee.phase4f_ssl_canary_v4 import run_phase4f_ssl_canary_v4

assert UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED is False
layout = KaggleLayout(output_dirname="rsna_unlabeled_image_ssl_canary_v4")
pre = preflight(layout=layout, output_dir=OUTPUT_DIR)
if not pre.ok or pre.data_root is None:
    raise SystemExit("DATA_NOT_AVAILABLE: attach rsna-knee-abnormality-detection")
cfg_path = str(Path(SOURCE_ROOT) / "configs" / "unlabeled_image_simsiam_ssl_canary_v4.yaml")
result = run_phase4f_ssl_canary_v4(
    data_root=pre.data_root,
    output_dir=OUTPUT_DIR,
    config_path=cfg_path,
    execute_training=True,
    allow_tiny_encoder_for_tests=False,
    internet_enabled=False,
    run_oof=False,
    run_full_unlabeled=False,
    segment="resume_to_end",
)
public = {
    "status": result.get("status"),
    "ok": result.get("ok"),
    "skip_reason": result.get("skip_reason"),
    "kernel_slug": result.get("kernel_slug"),
    "n_real_unlabelled_studies_pixels_opened": result.get("n_real_unlabelled_studies_pixels_opened"),
    "n_report_text_rows_processed": result.get("n_report_text_rows_processed"),
    "n_second_reader_rows_processed": result.get("n_second_reader_rows_processed"),
    "report_text_never_read": result.get("report_text_never_read"),
    "unlabeled_image_rows_opened_without_reading_report_text": result.get(
        "unlabeled_image_rows_opened_without_reading_report_text"
    ),
    "n_report_only_studies_processed": result.get("n_report_only_studies_processed"),
    "gpu_preflight_ok": result.get("gpu_preflight_ok"),
    "gpu_preflight": result.get("gpu_preflight"),
    "training_run": result.get("training_run"),
    "OOF_run": result.get("OOF_run"),
    "inference_run": result.get("inference_run"),
    "competition_submission_written": result.get("competition_submission_written"),
    "full_ssl_run_authorized": result.get("full_ssl_run_authorized"),
    "UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED": result.get("UNLABELED_IMAGE_SSL_EXECUTION_ALLOWED"),
    "completed_step": (result.get("training") or {}).get("completed_step"),
    "resumed_step": (result.get("training") or {}).get("resumed_step"),
    "precision_mode": result.get("precision_mode") or (result.get("training") or {}).get("precision_mode"),
    "applied_optimizer_steps": (result.get("training") or {}).get("applied_optimizer_steps"),
    "skipped_attempts": (result.get("training") or {}).get("skipped_attempts"),
    "amp_call_order": (result.get("training") or {}).get("amp_call_order"),
    "exception_diagnostics": result.get("exception_diagnostics"),
    "full_run_projection": result.get("full_run_projection"),
}
print("public=" + json.dumps(public, sort_keys=True))
print("PHASE4F4_CANARY status=" + str(result.get("status")))
print("DO_NOT_SUBMIT")
print("DO_NOT_LAUNCH_FULL_SSL")
if Path("/kaggle/working/submission.csv").exists():
    raise SystemExit("REFUSING_WORKING_ROOT_SUBMISSION_CSV")
if str(result.get("status")) not in {
    "SSL_CANARY_PASS_FULL_RUN_NOT_AUTHORIZED",
    "SSL_CANARY_FAIL",
    "SSL_CANARY_GPU_PREFLIGHT_FAIL",
}:
    raise SystemExit("UNEXPECTED_CANARY_STATUS:" + str(result.get("status")))
